In [1]:
#import libraries:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from scipy.stats import expon, reciprocal

from skopt.space import Real, Categorical, Integer
from skopt import gp_minimize

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from skopt.utils import use_named_args


from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import numpy as np
from scipy.stats import randint, uniform, loguniform
import joblib
import xgboost as xgb

In [2]:
#preprocessing steps:

pem = pd.read_csv('../data/pem_electrolyser_dataset.csv', encoding='ISO-8859-1')

# rename columns and set all to lowercase for consistency
pem.columns = pem.columns.str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

for col in pem.select_dtypes(include='object').columns: #assign category type to categorical features
    pem[col] = pem[col].astype('category')
    
#remove cell_design_type
pem.drop('cell_design_type', axis=1, inplace=True)


# Change flow_type to categorical as well
pem['flow_type_a_number'] = pem['flow_type_a_number'].astype('category')
pem['flow_type_c_number'] = pem['flow_type_c_number'].astype('category')

# avoid multicollinearity
pem.drop('water_flow_rateml/min', inplace=True, axis=1)
pem.drop('cathode_flow_areacm2', inplace=True, axis=1)

right_skewed_columns = ['cell_voltage', 'cell_current_desa/cm2', 'electrode_areacm2', 'anode_flow_areacm2', 'pressueratm']
for col in right_skewed_columns:
    # add 1 to all, to avoid log(0) error
    pem[col + '_log'] = np.log(pem[col] + 1)
    
# drop the original columns from the dataframe, so the log columns are the only ones that remain
pem.drop(right_skewed_columns, axis=1, inplace=True)


# remove power and powerw/cm2 from the pem dataframe
pem.drop(['powerw', 'powerw/cm2', 'electrolysis_type'], axis=1, inplace=True)



pem.head()


,anode_type,cathode_type,anode_gde,cathode_gde,membrane_type,cathode_catalyst,anode_catalyst,catholyte,anolyte,cell_design_number,hydrogen_flow_rate,temprature_cellk,electrode_shape,flow_type_a_number,flow_type_c_number,cell_voltage_log,cell_current_desa/cm2_log,electrode_areacm2_log,anode_flow_areacm2_log,pressueratm_log
0,carbon_plate,carbon_plate,porous_carbon_paper,porous_carbon_paper,nafion117,MoS2,MoS2,choline_0.5M,H2SO4_0.1mol,1,0.100,293.15,rectangular,1,1,1.308333,0.006940,2.302585,1.924249,0.693147
1,carbon_plate,carbon_plate,porous_carbon_paper,porous_carbon_paper,nafion117,MoS2,MoS2,choline_0.5M,H2SO4_0.1mol,1,0.675,293.15,rectangular,1,1,1.386294,0.014678,2.302585,1.924249,0.693147
2,carbon_plate,carbon_plate,porous_carbon_paper,porous_carbon_paper,nafion117,MoS2,MoS2,choline_0.5M,H2SO4_0.1mol,1,1.100,293.15,rectangular,1,1,1.458615,0.026071,2.302585,1.924249,0.693147
3,carbon_plate,carbon_plate,porous_carbon_paper,porous_carbon_paper,nafion117,MoS2,MoS2,choline_0.5M,H2SO4_0.1mol,1,1.900,293.15,rectangular,1,1,1.526056,0.042329,2.302585,1.924249,0.693147
4,carbon_plate,carbon_plate,porous_carbon_paper,porous_carbon_paper,nafion117,MoS2,MoS2,choline_0.5M,H2SO4_0.1mol,1,3.000,293.15,rectangular,1,1,1.589235,0.062438,2.302585,1.924249,0.693147


In [3]:
# load trained numeric rf model
rf_model_tuned = joblib.load('rf_tuned.joblib')
scaler = joblib.load('scaler_numeric.joblib')

In [4]:
    continuous_params_list = ['cell_design_number', 'temprature_cellk', 'cell_voltage_log', 'cell_current_desa/cm2_log',
                         'electrode_areacm2_log', 'anode_flow_areacm2_log', 'pressueratm_log']

In [5]:
space = [
    Real(1, 10, name='cell_design_number'),
    Real(293.15, 403.15, name='temprature_cellk'),
    Real(0.867100, 2.995732, name='cell_voltage_log'),
    Real(0.0002408010051107634, 1.0986122886681098, name ='cell_current_desa/cm2_log'),
    Real(2.302585092994046, 4.61512051684126, name='electrode_areacm2_log'),
    Real(1.6094379124341003, 3.9318256327243257, name='anode_flow_areacm2_log'),
    Real(0.6931471805599453, 1.3862943611198906, name='pressueratm_log'),
]

In [6]:
def objective(params):
    cell_design_number, temprature_cellk, cell_voltage_log, cell_current_desacm2_log, electrode_areacm2_log,
    anode_flow_areacm2_log, pressueratm_log = params
    input_data = pd.DataFrame({'cell_design_number': [cell_design_number],
                               'temperature_cellk': [temperature_cellk],
                               'cell_voltage_log': [cell_voltage_log],
                               'cell_current_desa/cm2_log': [cell_current_desacm2_log],
                               'electrode_areacm2_log': [electrode_areacm2_log],
                               'anode_flow_areacm2_log': [anode_flow_areacm2_log],
                               'pressueratm_log': [pressueratm_log]})
    scaled_input = scaler.transform(input_data)
    prediction = rf_model_tuned.predict(scaled_input)
    return -prediction[0]

In [7]:
# # Perform Bayesian Optimization
# result = gp_minimize(objective, space, n_calls=50, random_state=0)

# # Best parameter combination
# print("Best parameters: {}".format(result.x))

In [14]:

input_data = pd.DataFrame({'cell_design_number': [10],
                               'temprature_cellk': [337.03],
                               'cell_voltage_log': [2.9957319999999994],
                               'cell_current_desa/cm2_log': [1.0986122886681098],
                               'electrode_areacm2_log': [4.40519856592772],
                               'anode_flow_areacm2_log': [3.9318256327243257],
                               'pressueratm_log': [0.6931471805599453]})
scaled_input = scaler.transform(input_data)
prediction = rf_model_tuned.predict(scaled_input)
print(prediction)

print(scaled_input)

[5000.]
[[ 3.23041116 -0.05332644  2.95227022  2.51335684  1.38750351  1.57618193
  -0.51058866]]
